# 03 - Impulse Response Functions (IRF) and FEVD

This notebook covers **Impulse Response Functions** and **Forecast Error Variance
Decomposition** for VAR models using `chronobox`.

## Topics covered

1. Orthogonalized IRF (Cholesky decomposition)
2. Generalized IRF (Pesaran-Shin)
3. Bootstrap confidence intervals for IRF
4. Cumulative IRF
5. Forecast Error Variance Decomposition (FEVD)
6. Exercises

---

### Impulse Response Functions

The IRF traces the effect of a **one-time shock** to one variable on the current and
future values of all variables. For a VAR(p) model, the moving average representation is:

$$Y_t = \sum_{i=0}^{\infty} \Phi_i u_{t-i}$$

The IRF at horizon $h$ for a shock to variable $j$ on variable $i$ is $\Phi_h[i,j]$.

### Orthogonalized vs Generalized IRF

- **Orthogonalized (Cholesky):** Uses the Cholesky factor $P$ of $\Sigma_u = PP'$ to
  orthogonalize shocks. Results depend on variable ordering.
- **Generalized (Pesaran-Shin):** Does not require orthogonalization. Accounts for the
  correlation structure of shocks without imposing a causal ordering.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

from chronobox import VAR, IRF, FEVD

sys.path.insert(0, os.path.join("..", "utils"))
from plot_helpers import plot_irf, plot_fevd

%matplotlib inline
plt.rcParams["figure.dpi"] = 100
np.set_printoptions(precision=4, suppress=True)

print("All imports loaded successfully.")

## 1. Fitting the baseline VAR model

We use the Canadian macro dataset and fit a VAR(2) model as our baseline.

In [ ]:
# Load data and fit VAR
data_path = os.path.join("..", "data", "canada_macro.csv")
df = pd.read_csv(data_path)
var_names = ["e", "prod", "rw", "U"]
endog = df[var_names].values

model = VAR(lags=2, trend="c")
results = model.fit(endog, names=var_names)

print(f"VAR({results.k_ar}) fitted with {results.neqs} equations")
print(f"Stable: {results.is_stable}")
print(f"AIC: {results.aic:.4f}")

## 2. Orthogonalized IRF (Cholesky)

The Cholesky decomposition imposes a **recursive ordering**: the first variable is not
contemporaneously affected by any other; the second is affected only by the first; etc.

Our ordering: `e -> prod -> rw -> U`

This implies employment responds first and unemployment responds last within the same period.

In [ ]:
# Compute orthogonalized IRF with bootstrap confidence bands
irf_cholesky = results.irf(periods=20, method="cholesky", sigs=0.95, runs=500)

print(f"IRF shape: {irf_cholesky.irfs.shape}")
print(f"  (periods+1, K_response, K_shock) = ({irf_cholesky.irfs.shape[0]}, "
      f"{irf_cholesky.irfs.shape[1]}, {irf_cholesky.irfs.shape[2]})")
print(f"Bootstrap bands available: {irf_cholesky.lower is not None}")

In [ ]:
# Plot all orthogonalized IRFs using the built-in method
fig = irf_cholesky.plot(figsize=(16, 12))
plt.suptitle("Orthogonalized IRF (Cholesky) - 95% Bootstrap CI", fontsize=14, y=1.02)
plt.savefig(os.path.join("..", "outputs", "irf_cholesky_all.png"), bbox_inches="tight")
plt.show()

In [ ]:
# Focus on specific shock: effect of employment shock on all variables
fig = irf_cholesky.plot(impulse="e")
plt.suptitle("Response to Employment Shock (Cholesky)", fontsize=14, y=1.02)
plt.savefig(os.path.join("..", "outputs", "irf_employment_shock.png"), bbox_inches="tight")
plt.show()

**Economic interpretation:** A positive shock to employment (`e`):
- Immediately increases employment (by construction)
- Has a delayed positive effect on productivity as more workers enter the economy
- Eventually pushes up real wages as labour demand increases
- Reduces unemployment (negative response of U), with the effect building over several quarters

The bootstrap confidence bands indicate where the true response is likely to lie with 95% confidence.

## 3. Generalized IRF (Pesaran-Shin)

The **Generalized IRF** (Pesaran & Shin, 1998) does not require a specific ordering.
It conditions on the observed correlation structure of the shocks.

$$GIRF_i(h, \delta_j, \Omega_{t-1}) = E[Y_{t+h} | u_{jt} = \delta_j, \Omega_{t-1}] - E[Y_{t+h} | \Omega_{t-1}]$$

For a unit shock $\delta_j = \sqrt{\sigma_{jj}}$.

In [ ]:
# Compute generalized IRF
irf_generalized = results.irf(periods=20, method="generalized", sigs=0.95, runs=500)

fig = irf_generalized.plot(figsize=(16, 12))
plt.suptitle("Generalized IRF (Pesaran-Shin) - 95% Bootstrap CI", fontsize=14, y=1.02)
plt.savefig(os.path.join("..", "outputs", "irf_generalized_all.png"), bbox_inches="tight")
plt.show()

In [ ]:
# Compare Cholesky vs Generalized for a specific response
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

horizons = np.arange(irf_cholesky.irfs.shape[0])

# Response of U to prod shock
shock_idx = var_names.index("prod")
resp_idx = var_names.index("U")

ax = axes[0]
ax.plot(horizons, irf_cholesky.irfs[:, resp_idx, shock_idx], "b-", label="Cholesky", linewidth=1.5)
if irf_cholesky.lower is not None:
    ax.fill_between(horizons, irf_cholesky.lower[:, resp_idx, shock_idx],
                    irf_cholesky.upper[:, resp_idx, shock_idx], alpha=0.15, color="blue")
ax.axhline(0, color="k", linewidth=0.5)
ax.set_title("Cholesky: Response of U to prod shock")
ax.set_xlabel("Horizon")
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(horizons, irf_generalized.irfs[:, resp_idx, shock_idx], "r-", label="Generalized", linewidth=1.5)
if irf_generalized.lower is not None:
    ax.fill_between(horizons, irf_generalized.lower[:, resp_idx, shock_idx],
                    irf_generalized.upper[:, resp_idx, shock_idx], alpha=0.15, color="red")
ax.axhline(0, color="k", linewidth=0.5)
ax.set_title("Generalized: Response of U to prod shock")
ax.set_xlabel("Horizon")
ax.grid(True, alpha=0.3)

plt.suptitle("Cholesky vs Generalized IRF", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "irf_comparison.png"), bbox_inches="tight")
plt.show()

**Key difference:** The Cholesky IRF depends on the ordering of variables, while the
generalized IRF does not. When the contemporaneous correlations among shocks are high,
the two approaches can give very different results. The generalized IRF is preferred
when there is no strong theoretical reason to impose a recursive ordering.

## 4. Cumulative IRF

Cumulative IRFs show the **total accumulated effect** of a shock over time:

$$C_h = \sum_{s=0}^{h} \Phi_s$$

Useful for I(1) variables where we want to see the permanent effect of a shock.

In [ ]:
# Plot cumulative IRF
fig = irf_cholesky.plot_cum(figsize=(16, 12))
plt.suptitle("Cumulative Orthogonalized IRF", fontsize=14, y=1.02)
plt.show()

In [ ]:
# IRF to DataFrame for further analysis
irf_df = irf_cholesky.to_dataframe()
print(f"IRF DataFrame shape: {irf_df.shape}")
print(f"Columns: {list(irf_df.columns)}")
irf_df.head(10)

## 5. Forecast Error Variance Decomposition (FEVD)

FEVD tells us **what fraction** of the forecast error variance of each variable at
horizon $h$ is attributable to shocks in each variable:

$$\theta_{ij}(h) = \frac{\sum_{s=0}^{h-1} (e_i' \Phi_s P e_j)^2}{\sum_{s=0}^{h-1} e_i' \Phi_s \Sigma_u \Phi_s' e_i}$$

By construction, $\sum_j \theta_{ij}(h) = 1$ for all $i$ and $h$.

FEVD helps identify which shocks are the **main drivers** of variability for each variable.

In [ ]:
# Compute FEVD
fevd = results.fevd(periods=20, method="cholesky")

print(f"FEVD decomposition shape: {fevd.decomp.shape}")
print(f"  (periods+1, K_response, K_shock)")
print()
print(fevd.summary())

In [ ]:
# Plot FEVD for all variables
fig = fevd.plot(figsize=(12, 10))
plt.suptitle("Forecast Error Variance Decomposition", fontsize=14, y=1.02)
plt.savefig(os.path.join("..", "outputs", "fevd_all.png"), bbox_inches="tight")
plt.show()

In [ ]:
# FEVD table at selected horizons
fevd_df = fevd.to_dataframe()

for h in [1, 5, 10, 20]:
    print(f"\nFEVD at horizon {h}:")
    subset = fevd_df[fevd_df["horizon"] == h].pivot(
        index="response", columns="shock", values="fevd"
    )
    print(subset.round(4).to_string())

**Economic interpretation of FEVD:**

- At short horizons, each variable's forecast error is mostly driven by its own shocks
  (the diagonal elements are large).
- As the horizon increases, cross-variable contributions grow, revealing the transmission
  of shocks through the system.
- A variable whose forecast error is largely driven by *other* variables' shocks is
  relatively more "endogenous" in the system.
- A variable that explains a large share of other variables' forecast errors is a key
  driver of the macroeconomic dynamics.

In [ ]:
# FEVD using the plot_helpers utility
fig = plot_fevd(
    fevd.decomp,
    variable_names=var_names,
    title="FEVD - Stacked Area Chart"
)
plt.savefig(os.path.join("..", "outputs", "fevd_stacked.png"), bbox_inches="tight")
plt.show()

## Exercicio

### Exercicio 1: IRF para choques monetarios (US Macro)

Use o dataset `us_macro_quarterly.csv` com o ordenamento: `gdp -> inflation -> fed_funds -> unemployment`.

1. Ajuste um VAR(4)
2. Compute IRF ortogonalizada com 24 periodos e 500 bootstrap replications
3. Plote a resposta de GDP e unemployment a um choque em `fed_funds` (choque monetario)
4. Interprete: o aumento dos juros reduz GDP? Aumenta desemprego? Em quanto tempo?

**Dica:** Este e o exercicio classico de politica monetaria. Um choque positivo em
`fed_funds` representa um aperto monetario.

In [ ]:
# --- Exercicio 1: Seu codigo aqui ---

us_df = pd.read_csv(os.path.join("..", "data", "us_macro_quarterly.csv"))
us_names = ["gdp", "inflation", "fed_funds", "unemployment"]
us_endog = us_df[us_names].values

# Fit VAR(4)
us_model = VAR(lags=4, trend="c")
us_results = us_model.fit(us_endog, names=us_names)

# Orthogonalized IRF
us_irf = us_results.irf(periods=24, method="cholesky", sigs=0.95, runs=500)

# Plot response to fed_funds shock
fig = us_irf.plot(impulse="fed_funds")
plt.suptitle("Response to Monetary Policy Shock (Fed Funds)", fontsize=14, y=1.02)
plt.show()

# FEVD
us_fevd = us_results.fevd(periods=24)
fig = us_fevd.plot()
plt.suptitle("FEVD - US Macro", fontsize=14, y=1.02)
plt.show()

### Exercicio 2: Sensibilidade ao ordenamento

Reordene as variaveis canadenses como `U -> rw -> prod -> e` e recompute a IRF Cholesky.
Compare com a IRF original. O que muda? Por que a IRF generalizada e preferida quando
nao ha justificativa teorica para uma ordenacao especifica?

In [ ]:
# --- Exercicio 2: Seu codigo aqui ---

# Reorder variables
reorder_names = ["U", "rw", "prod", "e"]
reorder_endog = df[reorder_names].values

reorder_model = VAR(lags=2, trend="c")
reorder_results = reorder_model.fit(reorder_endog, names=reorder_names)

# Compare IRFs
irf_reorder = reorder_results.irf(periods=20, method="cholesky", sigs=0.95, runs=200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original: response of U to e shock
h = np.arange(irf_cholesky.irfs.shape[0])
axes[0].plot(h, irf_cholesky.irfs[:, 3, 0], "b-", linewidth=1.5)
axes[0].set_title("Original order: Response of U to e shock")
axes[0].axhline(0, color="k", linewidth=0.5)
axes[0].grid(True, alpha=0.3)

# Reordered: response of e to U shock
axes[1].plot(h, irf_reorder.irfs[:, 3, 0], "r-", linewidth=1.5)
axes[1].set_title("Reordered: Response of e to U shock")
axes[1].axhline(0, color="k", linewidth=0.5)
axes[1].grid(True, alpha=0.3)

plt.suptitle("Effect of Variable Ordering on Cholesky IRF", fontsize=14)
plt.tight_layout()
plt.show()

---

## Resumo

Neste notebook aprendemos:

1. **IRF ortogonalizada** usa a decomposicao de Cholesky e depende do ordenamento das variaveis
2. **IRF generalizada** (Pesaran-Shin) nao depende do ordenamento
3. **Bootstrap** fornece intervalos de confianca para a IRF
4. **IRF cumulativa** mostra o efeito acumulado de um choque ao longo do tempo
5. **FEVD** decompoe a variancia do erro de previsao em contribuicoes de cada variavel
6. A FEVD revela quais choques sao os principais drivers da variabilidade no sistema

No proximo notebook, testaremos **causalidade de Granger** entre as variaveis.